# NB11 — Track B LOGO (all 6 generators) + matched neural-only baseline

Full leave-one-generator-out for the joint fine-tuned hybrid: for each of the 6 held-out generators,
**retrain from scratch** excluding that generator's AI (train+val), then evaluate on its test-split AI +
all test humans. Same K=9 + fused-standardizer policy as NB10.

**Action #1 built in:** each fold ALSO trains a **matched neural-only baseline** (768-only, standardized
the SAME way) so the fusion lift is measured on an apples-to-apples footing — no raw-vs-scaled inflation.

**Runtime is large** (6 folds × 2 models × full fine-tune) — user accepted this. Full checkpoint/resume:
progress is saved per (fold, model); a crashed commit re-run skips finished cells and resumes the
in-flight train from its last step. Results accumulate in a JSON so partial runs aren't lost.

## 1 · Config

In [1]:
import os, json, math, random, numpy as np, pandas as pd, torch
P_DATASET  = "/kaggle/input/notebooks/bahaaqassem/nb3-build-dataset/dataset.parquet"           # EDIT
P_VSTAT16  = "/kaggle/input/notebooks/bahaaqassem/ph2-nb6e-extract-11-features/vstat16_scaled.parquet"    # EDIT
CKPT_DIR   = "/kaggle/working/logo_ckpt"
RESUME_FROM= "/kaggle/input/datasets/bahaaqassem/trackb-logo-all-generators/logo_ckpt"                    # EDIT after a crash; auto-ignored if absent
RESULTS    = "/kaggle/working/logo_results.json"
RESUME_RES = "/kaggle/input/datasets/bahaaqassem/trackb-logo-all-generators/logo_results.json"

MODEL_ID  = "CAMeL-Lab/bert-base-arabic-camelbert-msa"
STAT_COLS = ["burstiness","ttr","quote_ratio","function_word_ratio","compressibility"]
GENERATORS= ["deepseek","sonnet","qwen","gemini","gpt","opus"]
K_CHUNKS, MAX_CT, STRIDE = 9, 510, 460
HIDDEN, DROPOUT = 256, 0.0
LR_ENC, LR_HEAD, WD = 2e-5, 1e-3, 0.01
EPOCHS, MICRO_BS, GRAD_ACCUM, SAVE_EVERY = 3, 2, 16, 200
SEED = 42
os.makedirs(CKPT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("config loaded | device", DEV)

config loaded | device cuda


## 2 · Data + cached chunk ids (reuses NB10's chunks_K9.npz if provided)

In [2]:
df = pd.read_parquet(P_DATASET)
if "article_id" in df.columns: df=df.set_index("article_id")
v16=pd.read_parquet(P_VSTAT16)
if "article_id" in v16.columns: v16=v16.set_index("article_id")
df=df.loc[df.index.intersection(v16.index)]; v16=v16.loc[df.index]
Xstat=v16[STAT_COLS].to_numpy(np.float32)
y=df["label"].to_numpy(np.int64); gen=df["generator"].fillna("__human__").to_numpy()
split=df["split"].to_numpy(); texts=df["text"].astype(str).tolist()

!pip install -q transformers
from transformers import AutoTokenizer
tok=AutoTokenizer.from_pretrained(MODEL_ID)
CLS,SEP,PAD=tok.cls_token_id,tok.sep_token_id,tok.pad_token_id; CHUNK_LEN=MAX_CT+2

def to_chunks(t):
    ids=tok(t,add_special_tokens=False)["input_ids"] or [tok.unk_token_id]
    wins=[ids[i:i+MAX_CT] for i in range(0,len(ids),STRIDE)][:K_CHUNKS] or [ids[:MAX_CT]]
    return [[CLS]+w+[SEP]+[PAD]*(CHUNK_LEN-2-len(w)) for w in wins]

CACHE="/kaggle/working/chunks_K9.npz"
SRC = CACHE if os.path.exists(CACHE) else ("/kaggle/input/nb10-chunks/chunks_K9.npz"
        if os.path.exists("/kaggle/input/nb10-chunks/chunks_K9.npz") else None)
if SRC:
    z=np.load(SRC, allow_pickle=True); CH=z["ch"]; NCH=z["nch"]; print("loaded cached chunks from", SRC)
else:
    from tqdm.auto import tqdm
    CH=np.full((len(texts),K_CHUNKS,CHUNK_LEN),PAD,np.int32); NCH=np.zeros(len(texts),np.int8)
    for i,t in enumerate(tqdm(texts,desc="chunking")):
        ch=to_chunks(t); NCH[i]=len(ch)
        for j,c in enumerate(ch): CH[i,j]=c
    np.savez_compressed(CACHE,ch=CH,nch=NCH); print("built chunks")
print("chunks:", CH.shape)

config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

chunking:   0%|          | 0/7101 [00:00<?, ?it/s]

built chunks
chunks: (7101, 9, 512)


## 3 · Models (hybrid + matched neural-only), datasets, standardizer

In [3]:
import torch.nn as nn
from transformers import AutoModel
from torch.utils.data import DataLoader, Dataset

class Net(nn.Module):
    def __init__(self, stat_dim, use_stat):
        super().__init__()
        self.use_stat=use_stat
        self.enc=AutoModel.from_pretrained(MODEL_ID); self.enc.gradient_checkpointing_enable()
        h=self.enc.config.hidden_size; fused=h+(stat_dim if use_stat else 0)
        self.register_buffer("mu",torch.zeros(fused)); self.register_buffer("sd",torch.ones(fused))
        self.head=nn.Sequential(nn.Linear(fused,HIDDEN),nn.LayerNorm(HIDDEN),nn.ReLU(),
                                nn.Dropout(DROPOUT),nn.Linear(HIDDEN,2))
    def encode(self,ids,nch):
        B,K,L=ids.shape; flat=ids.view(B*K,L); att=(flat!=PAD).long()
        cls=self.enc(input_ids=flat,attention_mask=att).last_hidden_state[:,0,:].view(B,K,-1)
        m=(torch.arange(K,device=ids.device)[None,:]<nch[:,None]).float().unsqueeze(-1)
        return (cls*m).sum(1)/m.sum(1).clamp(min=1)
    def forward(self,ids,nch,stat):
        v=self.encode(ids,nch)
        if self.use_stat: v=torch.cat([v,stat],1)
        v=(v-self.mu)/self.sd; return self.head(v)

class DS(Dataset):
    def __init__(self,rows): self.rows=rows
    def __len__(self): return len(self.rows)
    def __getitem__(self,k):
        i=self.rows[k]
        return (torch.from_numpy(CH[i].astype(np.int64)),int(NCH[i]),
                torch.from_numpy(Xstat[i]),int(y[i]),i)
def collate(b):
    return (torch.stack([x[0] for x in b]), torch.tensor([x[1] for x in b]),
            torch.stack([x[2] for x in b]), torch.tensor([x[3] for x in b]), [x[4] for x in b])

@torch.no_grad()
def fit_std(model, rows):
    model.eval(); s=s2=None; n=0
    for ids,nch,st,yy,idx in DataLoader(DS(rows),batch_size=8,collate_fn=collate):
        ids,nch,st=ids.to(DEV),nch.to(DEV),st.to(DEV)
        v=model.encode(ids,nch)
        if model.use_stat: v=torch.cat([v,st],1)
        s=v.sum(0) if s is None else s+v.sum(0); s2=(v*v).sum(0) if s2 is None else s2+(v*v).sum(0); n+=v.shape[0]
    mu=s/n; sd=torch.sqrt(torch.clamp(s2/n-mu*mu,min=1e-6)); model.mu.copy_(mu); model.sd.copy_(sd)
print("models ready")

models ready


## 4 · Train one fold-model (with per-model checkpoint/resume)

In [4]:
from sklearn.utils.class_weight import compute_class_weight
from transformers import get_linear_schedule_with_warmup

def ckpt_path(tag): return os.path.join(CKPT_DIR, f"{tag}.pt")
def save_ck(tag,model,opt,sched,scaler,ep,step):
    torch.save({"model":model.state_dict(),"opt":opt.state_dict(),
                "sched":sched.state_dict(),"scaler":scaler.state_dict(),"ep":ep,"step":step,
                "trng":torch.get_rng_state(),"crng":torch.cuda.get_rng_state_all(),
                "nrng":np.random.get_state(),"prng":random.getstate()}, ckpt_path(tag))
def resume_path(tag):
    for base in (CKPT_DIR, RESUME_FROM):
        p=os.path.join(base,f"{tag}.pt")
        if os.path.exists(p): return p
    return None

def train_fold(tag, use_stat, train_rows):
    model=Net(len(STAT_COLS),use_stat).to(DEV)
    cw=compute_class_weight("balanced",classes=np.array([0,1]),y=y[train_rows])
    lossf=nn.CrossEntropyLoss(weight=torch.tensor(cw,dtype=torch.float32,device=DEV))
    dl=DataLoader(DS(train_rows),batch_size=MICRO_BS,shuffle=True,collate_fn=collate,drop_last=True)
    total=(len(dl)//GRAD_ACCUM)*EPOCHS
    enc=[p for n,p in model.named_parameters() if n.startswith("enc.")]
    hd =[p for n,p in model.named_parameters() if not n.startswith("enc.")]
    opt=torch.optim.AdamW([{"params":enc,"lr":LR_ENC},{"params":hd,"lr":LR_HEAD}],weight_decay=WD)
    sched=get_linear_schedule_with_warmup(opt,int(0.06*total),total)
    scaler=torch.amp.GradScaler('cuda')
    rp=resume_path(tag); start_ep=gstep=0
    if rp:
        ck=torch.load(rp,map_location=DEV); model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        torch.set_rng_state(ck["trng"].cpu()); torch.cuda.set_rng_state_all([s.cpu() for s in ck["crng"]])
        np.random.set_state(ck["nrng"]); random.setstate(ck["prng"]); start_ep,gstep=ck["ep"],ck["step"]
        print(f"  resume {tag}: ep{start_ep} step{gstep}")
    else:
        fit_std(model,train_rows); save_ck(tag,model,opt,sched,scaler,0,0)
    model.train()
    for ep in range(start_ep,EPOCHS):
        opt.zero_grad()
        for bi,(ids,nch,st,yy,idx) in enumerate(dl):
            ids,nch,st,yy=ids.to(DEV),nch.to(DEV),st.to(DEV),yy.to(DEV)
            with torch.amp.autocast('cuda'):
                loss=lossf(model(ids,nch,st),yy)/GRAD_ACCUM
            scaler.scale(loss).backward()
            if (bi+1)%GRAD_ACCUM==0:
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(); gstep+=1
                if gstep%SAVE_EVERY==0: save_ck(tag,model,opt,sched,scaler,ep,gstep)
        save_ck(tag,model,opt,sched,scaler,ep+1,gstep)
    return model
print("train_fold ready")

train_fold ready


## 5 · Run LOGO over all 6 generators (hybrid + matched neural-only), resumable

In [5]:
from sklearn.metrics import f1_score

def load_results():
    for p in (RESULTS, RESUME_RES):
        if os.path.exists(p):
            return json.load(open(p))
    return {}
def save_results(r): json.dump(r, open(RESULTS,"w"), indent=2)

@torch.no_grad()
def macro_f1(model, rows):
    model.eval(); P=[]; Y=[]
    for ids,nch,st,yy,idx in DataLoader(DS(rows),batch_size=8,collate_fn=collate):
        ids,nch,st=ids.to(DEV),nch.to(DEV),st.to(DEV)
        with torch.amp.autocast('cuda'):
            P.append(model(ids,nch,st).argmax(1).cpu().numpy())
        Y.append(yy.numpy())
    return 100*f1_score(np.concatenate(Y),np.concatenate(P),average="macro")

results=load_results()
human=(y==0); is_tv=np.isin(split,["train","val"]); is_test=(split=="test")
for g in GENERATORS:
    tr=np.where(is_tv & (human | (gen!=g)))[0].tolist()
    te=np.where(is_test & (human | (gen==g)))[0].tolist()
    for use_stat,mname in [(True,"hybrid"),(False,"neural")]:
        key=f"{g}:{mname}"
        if key in results: print(f"skip {key} = {results[key]:.2f}"); continue
        print(f"=== training {key} (train {len(tr)}, test {len(te)}) ===")
        m=train_fold(f"{g}_{mname}", use_stat, tr)
        results[key]=macro_f1(m, te); save_results(results)
        print(f"  {key} test-fold MacroF1 = {results[key]:.2f}")
        del m; torch.cuda.empty_cache()
print("all folds done")

skip deepseek:hybrid = 98.83
skip deepseek:neural = 99.53
skip sonnet:hybrid = 99.77
skip sonnet:neural = 99.54
skip qwen:hybrid = 99.34
skip qwen:neural = 99.34
skip gemini:hybrid = 99.56
=== training gemini:neural (train 5630, test 602) ===


pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-msa
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`use_cache=True` is incompatible with gradient checkpointing. Sett

...

BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-msa
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  opus:hybrid test-fold MacroF1 = 99.56
=== training opus:neural (train 5691, test 602) ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-msa
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  opus:neural test-fold MacroF1 = 99.56
all folds done


## 6 · Summary — hybrid vs matched neural-only (the clean fusion-lift number)

In [6]:
rows=[]
for g in GENERATORS:
    h=results.get(f"{g}:hybrid"); n=results.get(f"{g}:neural")
    rows.append({"held_out":g,"hybrid":round(h,2) if h else None,
                 "neural":round(n,2) if n else None,
                 "lift": round(h-n,2) if (h and n) else None})
S=pd.DataFrame(rows); import IPython.display as ipd; ipd.display(S)
hv=[r["hybrid"] for r in rows if r["hybrid"]]; nv=[r["neural"] for r in rows if r["neural"]]
if hv and nv:
    print(f"HYBRID   LOGO  mean {np.mean(hv):.2f}  worst {min(hv):.2f}")
    print(f"NEURAL   LOGO  mean {np.mean(nv):.2f}  worst {min(nv):.2f}   (matched standardizer)")
    print(f"FUSION LIFT (matched): mean {np.mean(hv)-np.mean(nv):+.2f}  worst {min(hv)-min(nv):+.2f}pp")
    print("This is the honest, apples-to-apples fusion-lift number for the thesis (no raw-vs-scaled inflation).")

,held_out,hybrid,neural,lift
0,deepseek,98.83,99.53,-0.70
1,sonnet,99.77,99.54,0.23
2,qwen,99.34,99.34,0.00
3,gemini,99.56,99.56,0.00
4,gpt,99.18,98.36,0.82
5,opus,99.56,99.56,0.00


HYBRID   LOGO  mean 99.37  worst 98.83
NEURAL   LOGO  mean 99.31  worst 98.36   (matched standardizer)
FUSION LIFT (matched): mean +0.06  worst +0.47pp
This is the honest, apples-to-apples fusion-lift number for the thesis (no raw-vs-scaled inflation).
